<a href="https://colab.research.google.com/github/jiedeng99/ml/blob/main/notebooks/cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!gdown '1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy' --output food-11.zip
!unzip food-11.zip

Streaming output truncated to the last 5000 lines.
  inflating: food-11/training/labeled/01/1_284.jpg  
  inflating: food-11/training/labeled/01/1_139.jpg  
  inflating: food-11/training/labeled/01/1_337.jpg  
  inflating: food-11/training/labeled/01/1_264.jpg  
  inflating: food-11/training/labeled/01/1_166.jpg  
  inflating: food-11/training/labeled/01/1_276.jpg  
  inflating: food-11/training/labeled/01/1_154.jpg  
  inflating: food-11/training/labeled/01/1_293.jpg  
  inflating: food-11/training/labeled/01/1_233.jpg  
  inflating: food-11/training/labeled/01/1_112.jpg  
  inflating: food-11/training/labeled/01/1_30.jpg  
  inflating: food-11/training/labeled/01/1_257.jpg  
  inflating: food-11/training/labeled/01/1_260.jpg  
  inflating: food-11/training/labeled/01/1_203.jpg  
  inflating: food-11/training/labeled/01/1_338.jpg  
  inflating: food-11/training/labeled/01/1_310.jpg  
  inflating: food-11/training/labeled/01/1_313.jpg  
  inflating: food-11/training/labeled/01/1_342.jp

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision.datasets import DatasetFolder
from tqdm.auto import tqdm

In [ ]:
train_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    # Try add some transforms here later
    transforms.ToTensor(),
])

test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [ ]:
from torch.utils.data.dataset import Dataset
batch_size = 128

train_set = DatasetFolder("food-11/training/labeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
valid_set = DatasetFolder("food-11/validation", loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)
unlabeled_set = DatasetFolder("food-11/training/unlabeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
test_set = DatasetFolder("food-11/testing", loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

In [ ]:
class Classifier(nn.Module):
  def __init__(self):
    super().__init__()

    # CNN layers: extract features
    self.cnn_layers = nn.Sequential(
        nn.Conv2d(3, 64, 3, 1, 1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2, 2, 0),

        nn.Conv2d(64, 128, 3, 1, 1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2, 2, 0),

        nn.Conv2d(128, 256, 3, 1, 1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(4, 4, 0),
    )

    # Fully connected layers: take the flattened features from the CNN part and perform classification
    self.fc_layers = nn.Sequential(
        nn.Linear(256 * 8 * 8, 256),
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Linear(256, 11)
    )

  def forward(self, x):
    # input (x): [batch_size, 3, 128, 128]
    # output: [batch_size, 11]

    # Extract features by convolutional layers
    x = self.cnn_layers(x)

    # The extracted feature map must be flatten before going to fully-connected layers
    x = x.flatten(1)

    # The features are transformed by fully-connected layers to obtain the final logits
    x = self.fc_layers(x)
    return x

In [ ]:
def get_pseudo_labels(dataset, model, threshold=0.65):
  device = "cuda" if torch.cuda.is_available() else "cpu"

  data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

  model.eval()
  softmax = nn.Softmax(dim=-1)

  for batch in tqdm(data_loader):
    img, _ = batch

    with torch.no_grad():
      logits = model(img.to(device))
    probs = softmax(logits)

  # ToDo: Filter the data and construct a new dataset
  model.train()
  return dataset

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Classifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-5)

n_epochs = 40

do_semi = False

for epoch in range(n_epochs):
  if do_semi:
    # semi-supervised learning
    pseudo_set = get_pseudo_labels(unlabeled_set, model)
    concat_set = ConcatDataset([train_set, pseudo_set])
    train_loader = DataLoader(concat_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

  model.train()

  train_loss = []
  train_accs = []

  for batch in tqdm(train_loader):
    imgs, labels = batch
    imgs, labels = imgs.to(device), labels.to(device)

    logits = model(imgs)

    loss = criterion(logits, labels)

    optimizer.zero_grad()

    loss.backward()   # Calculate gradients

    # Clip the gradient norms for stable training
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)

    optimizer.step()  # Update the parameters with computed gradients

    acc = (logits.argmax(dim=-1) == labels).float().mean()

    train_loss.append(loss.item())
    train_accs.append(acc)

  train_loss = sum(train_loss) / len(train_loss)
  train_acc = sum(train_accs) / len(train_accs)

  # Print the information for each epoch
  print(f"[Train | {epoch + 1:03d}/{n_epochs:03d}] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

  model.eval()
  valid_loss = []
  valid_accs = []

  for batch in tqdm(valid_loader):
    imgs, labels = batch
    imgs, labels = imgs.to(device), labels.to(device)

    with torch.no_grad():
      logits = model(imgs)
    loss = criterion(logits, labels)

    acc = (logits.argmax(dim=-1) == labels).float().mean()

    valid_loss.append(loss.item())
    valid_accs.append(acc)

  valid_loss = sum(valid_loss) / len(valid_loss)
  valid_acc = sum(valid_accs) / len(valid_accs)

  print(f"[Valid | {epoch + 1:03d}/{n_epochs:03d}] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")



  0%|          | 0/25 [00:00<?, ?it/s]

[Train | 001/040] loss = 2.30108, acc = 0.19906


  0%|          | 0/6 [00:00<?, ?it/s]

[Valid | 001/040] loss = 2.81362, acc = 0.10703


  0%|          | 0/25 [00:00<?, ?it/s]

...

[Valid | 039/040] loss = 2.59164, acc = 0.48464


  0%|          | 0/25 [00:00<?, ?it/s]

[Train | 040/040] loss = 0.00410, acc = 1.00000


  0%|          | 0/6 [00:00<?, ?it/s]

[Valid | 040/040] loss = 2.55200, acc = 0.51927


In [ ]:
model_path = 'food_classifier.pth'
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

# To load the model, you'll need to create a new instance of the Classifier class
# and then load the state dictionary.
loaded_model = Classifier().to(device) # Assuming 'device' is still defined
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.eval() # Set the model to evaluation mode
print(f"Model loaded from {model_path}")


Model saved to food_classifier.pth
Model loaded from food_classifier.pth


In [ ]:
model.eval()

predictions = []

for batch in tqdm(test_loader):
  imgs, labels = batch
  imgs = imgs.to(device)

  with torch.no_grad():
    logits = model(imgs)

  predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

with open("predict.csv", "w") as f:
  f.write("Id,Category\n")
  for i, label in enumerate(predictions):
    f.write(f"{i},{label}\n")

  0%|          | 0/27 [00:00<?, ?it/s]